# **COMPUTATIONAL THINKING**
## Name: Truong Le Vu Hoang
## Student ID: 24120056
## Class: 24CTT2/1

# H06: Building a Todo Application with Streamlit, FastAPI, and Firebase

This notebook documents the process of building a Todo application using Streamlit for the frontend, FastAPI for the backend, Firebase Authentication for user login, and Cloud Firestore for  storage.


## Important Note:
To prevent secret keys and api exposure, this project only covers the source code, example configuration, and web deploy url

## Web URL: https://computational-thinking-todo-app.onrender.com/

## 1. Objectives

The application must support the core requirements below:

- Register and log in with email/password.
- Create, view, update, and delete Todo items.
- Only display Todo items that belong to the currently authenticated user.

Additional features implemented:

- Search Todo items by title or description.
- Filter Todo items by status and priority.
- Basic role-based access control with `user`, `guest` and `admin` roles.
- Google sign-in integration.


## 2. Folder Structure

```text
todo-app/
|-- backend/
|   |-- main.py
|   `-- app/
|       |-- config.py
|       |-- firebase.py
|       |-- auth.py
|       |-- schemas.py
|       `-- routes/
|           |-- users.py
|           `-- todos.py
|-- frontend/
|   |-- app.py
|   |-- config.py
|   |-- firebase_auth.py
|   |-- api_client.py
|   |-- session.py
|   `-- components.py
|   
|-- .env.example
|-- requirements.txt
|-- .gitignore
|-- README.md
|-- firebase-key-example.json
`-- procedure.ipynb
```


## 3. Core Architecture

The system has four main parts:

- **Streamlit frontend**: renders login forms, Todo forms, filters, and dashboard controls.
- **Firebase Authentication**: authenticates users and returns a Firebase ID token, support Google sign in using OAuth2
- **FastAPI backend**: verifies Firebase ID tokens, applies authorization rules, and exposes Todo REST endpoints.
- **Cloud Firestore**: stores users and Todo documents.



## 4. Authentication Flow

### Login using Email & Password
1. The user selects the **"Login"** or **"Register"** action and enters their email and password in the main login form.
2. Streamlit calls the Firebase Authentication REST API (via `firebase_signin` or `firebase_signup`).
3. Firebase validates the credentials and returns an `idToken` if successful.
4. Streamlit stores the `idToken` and initializes the user's metadata in `st.session_state.auth` (including `email`, `role`, `uid`, and sets `is_guest` to `False`).
5. Streamlit fetches the latest user profile from the backend to ensure data consistency.
6. For every subsequent backend request, Streamlit attaches the token in the HTTP header:
```http Authorization: Bearer <Firebase idToken>```

7. FastAPI backend uses the Firebase Admin SDK to verify the token. If valid, FastAPI extracts the user's uid and processes the request.



### Alternative Login Methods
* **Google Sign-in:** Users can authenticate via Google OAuth2. Streamlit handles the redirect, extracts the authorization code, and exchanges it for a Firebase `idToken`.
* **Guest Mode:** Users can test the application without an account. Streamlit assigns a temporary session (`is_guest = True`) and manages data purely in the local RAM (`st.session_state.guest_todos`). No requests are sent to the backend, and data is cleared upon refresh.
* **Admin Access:** A backdoor login using a specific master password that grants local admin privileges (`role = "admin"`) for testing purposes.

## 5. Firestore Data Design

Collection `users` stores role metadata. The document id is the Firebase `uid`.

```json
{
  "role": "user",
  "email_verified": false,
  "created_at": "2026-05-13T..."
}
```

Collection `todos` stores Todo items.

```json
{
  "title": "Study FastAPI",
  "description": "Finish the Todo assignment",
  "due_date": "2026-05-20",
  "priority": "normal",
  "done": false,
  "owner_uid": "firebase-user-uid",
  "created_at": "2026-05-13T...",
  "updated_at": "2026-05-13T..."
}
```



## 6. Backend Design

The backend uses FastAPI and Firebase Admin SDK.

- `backend/main.py` creates the FastAPI app and includes routers.
- `backend/app/config.py` reads environment variables.
- `backend/app/firebase.py` initializes Firebase Admin SDK and Firestore.
- `backend/app/auth.py` verifies Firebase ID tokens and creates a user document if needed.
- `backend/app/schemas.py` defines Pydantic models.
- `backend/app/routes/todos.py` contains Todo CRUD endpoints.
- `backend/app/routes/users.py` contains `/me` and admin role management.

Main endpoints:

- `GET /me`: return the current user's uid, email, and role.
- `GET /todos`: list Todo items with optional search and filters.
- `POST /todos`: create a Todo item for the current user.
- `PUT /todos/{todo_id}`: update an existing Todo item.
- `DELETE /todos/{todo_id}`: delete a Todo item.
- `POST /users/{target_uid}/role`: allow an admin to change another user's role.


## 7. Frontend Design

The frontend uses Streamlit.

- `frontend/app.py` is the entrypoint and controls page flow.
- `frontend/config.py` reads frontend environment variables.
- `frontend/firebase_auth.py` calls Firebase Authentication REST endpoints.
- `frontend/api_client.py` calls the FastAPI backend.
- `frontend/session.py` manages Streamlit authentication session state.
- `frontend/components.py` renders login, Todo dashboard, Todo editor, filters, and admin UI.

The frontend keeps the Firebase ID token in session state and attaches it to backend requests. It never uses the Firebase service account key.


## 8. Firebase Setup
If you want to run the app yourself, follow these steps and fill in the ``.env.example `` and ``firebase-key-example.json`` files  


Required Firebase steps:

1. Create a Firebase project.
2. Enable Email/Password sign-in in Firebase Authentication.
3. Optionally enable Google sign-in for the advanced feature.
4. Create a Cloud Firestore database.
5. Copy the Firebase Web API key for the frontend.
6. Create a service account JSON file for the backend.

## 9. Environment Variables

Use `.env.example` as the template and create a local `.env` file.

Important variables:

- `GOOGLE_APPLICATION_CREDENTIALS`: path to the backend service account JSON file.
- `FIREBASE_PROJECT_ID`: Firebase project id.
- `BACKEND_CORS_ORIGINS`: allowed Streamlit origins for FastAPI CORS.
- `FIREBASE_API_KEY`: Firebase Web API key used by Firebase Authentication REST API.
- `API_BASE_URL`: FastAPI backend URL.
- `GOOGLE_OAUTH_CLIENT_ID`, `GOOGLE_OAUTH_CLIENT_SECRET`, `GOOGLE_REDIRECT_URI`: optional Google login settings.



## 10. Running Locally

Install and run the backend:

```bash
pip install -r backend/requirements.txt
uvicorn backend.main:app --reload --host 0.0.0.0 --port 8000
```

Install and run the frontend:

```bash
pip install -r frontend/requirements.txt
streamlit run frontend/app.py
```

Then open `http://localhost:8501`.


## 11. Artificial Intelligence Usage Note

Since this project requires immense foundation in `Python` and `Web Development`, which are challenging for a sophomore. To help with the workload and learn faster, I used AI tools to assist with writing code, fixing bugs, and testing the application.

Tool used: 

`Codex 5.5`  
`Gemini 3.1 Pro`  
`Claude Haiku 4.5`


